# 🐉 Imagen → 3D con Hunyuan3D-2 (gratis) — versión con TODOS los parchesEste notebook trae **cada parche** que fuimos encontrando al hacerlo funcionar de verdad en Colab, en celdas separadas para ir paso a paso:| Problema real | Parche (celda) ||---|---|| `AttributeError … _blas_supports_fpe` al importar | **Celda 1B**: `numpy>=2.1` + **Reiniciar sesión** || Warnings `numba / cuml / cudf incompatible` | inofensivos, se ignoran (Celda 1B) || `Model path not exists` con el **mini** | **Celda 3**: usar modelo **completo** `tencent/Hunyuan3D-2` || El modelo se **buguea / sale deforme con fondo** | **Celda 2B**: quitar el fondo SÍ o SÍ (imagen transparente) || `CUDA out of memory` | **Celda 3B**: bajar `octree_resolution` a 192 |> ⚠️ **Regla de oro:** la imagen tiene que entrar **sin fondo** (PNG transparente). La Celda 2B lo hace sola; si igual sale raro, subí un PNG ya recortado.## Orden de ejecución1. **GPU T4**: `Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU** → Guardar.2. **Celda 1** (instalar) → **Celda 1B** (parche numpy) → **Reiniciar sesión**.3. **Celda 2** (subir imagen) → **Celda 2B** (quitar fondo).4. **Celda 3** (generar). Si da *out of memory* → **Celda 3B**.5. **Celda 4** (descargar `.glb`).

## Celda 1 — Instalar Hunyuan3D-2Tarda ~5–8 min la primera vez. Instala solo lo necesario para la **forma** (no compila los módulos CUDA de textura, que fallan seguido en Colab).

In [ ]:
!nvidia-smi -Limport osos.chdir('/content')if not os.path.isdir('/content/Hunyuan3D-2'):    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.gitos.chdir('/content/Hunyuan3D-2')# Dependencias de la generacion de FORMA!pip install -q ninja!pip install -q diffusers transformers accelerate trimesh omegaconf einops opencv-python-headless huggingface_hub!pip install -q rembg onnxruntime            # para quitar el fondo (Celda 2B)!pip install -q -e . 2>&1 | tail -3          # instala el paquete hy3dgen (ignora warnings de version)import torchprint('\ntorch:', torch.__version__, '| GPU:', torch.cuda.is_available())print('✅ Celda 1 lista. Ahora corré la Celda 1B (parche numpy).')

## Celda 1B — Parche `numpy` (arregla `scipy _blas_supports_fpe`) → **REINICIAR SESIÓN**Sin esto, la Celda 3 tira `AttributeError: _blas_supports_fpe` al importar `trimesh`/`hy3dgen` (a `scipy` le falta ese símbolo en numpy < 2.1).Los warnings de `numba / cuml / cudf incompatible` que aparezcan son **inofensivos** (son de RAPIDS, no los usamos).

In [ ]:
# Alinea numpy para que scipy no rompa!pip install -q -U "numpy>=2.1"print('\n⚠️  AHORA hacé:  Entorno de ejecución → Reiniciar sesión')print('   (obligatorio para que tome el numpy nuevo). Después seguí con la Celda 2.')print('   Los warnings numba/cuml/cudf de arriba son inofensivos, ignoralos.')

## Celda 2 — Subir tu imagenCualquier imagen sirve; la Celda 2B le quita el fondo. Mejor si el personaje está **de frente** y entero.> Si ya tenés un **PNG con fondo transparente**, subilo igual y podés saltear la Celda 2B.

In [ ]:
from google.colab import filesfrom PIL import Imageup = files.upload()IMG = list(up.keys())[0]im = Image.open(IMG)print('Imagen:', IMG, '| modo:', im.mode, '| tamaño:', im.size)print('✅ Subida. Seguí con la Celda 2B para quitar el fondo.')

## Celda 2B — Quitar el fondo (SÍ o SÍ)Deja la imagen **transparente** (RGBA con fondo recortado). Esto es lo que evita que el modelo se **bugee o salga deforme** (interpreta el fondo como parte del cuerpo).Genera `/content/entrada_sin_fondo.png` y apunta `IMG` a ese archivo.

In [ ]:
from PIL import Imageimport numpy as npimg = Image.open(IMG)need = Trueif img.mode == 'RGBA':    a = np.array(img.split()[-1])    if (a < 250).mean() > 0.05:   # ya tiene transparencia real        need = False        print('La imagen ya viene sin fondo ✅')if need:    from rembg import remove    img = remove(img.convert('RGBA'))   # quita el fondo -> RGBA transparente    print('Fondo quitado ✅')IMG = '/content/entrada_sin_fondo.png'img.convert('RGBA').save(IMG)print('Imagen lista (sin fondo):', IMG, '| modo:', img.mode)

## Celda 3 — Generar el modelo 3D (forma)Usa el modelo **completo** `tencent/Hunyuan3D-2` (el `mini` da `Model path not exists` porque su subfolder no coincide con el que busca el código).La **primera vez** baja ~10 GB de pesos → paciencia (vas a ver barras de descarga). Después genera la malla (~30–60 s en la T4).

In [ ]:
import os, torchos.chdir('/content/Hunyuan3D-2')from PIL import Imagefrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipelineprint('Cargando el modelo (la 1a vez baja ~10 GB)...')pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')print('✅ Modelo cargado.')img = Image.open(IMG).convert('RGBA')   # IMG ya viene sin fondo desde la Celda 2Bmesh = pipe(image=img, num_inference_steps=30, octree_resolution=256,            generator=torch.manual_seed(0))[0]OUT = '/content/hunyuan_mesh.glb'mesh.export(OUT)print('\nResultado:', ('✅ ' + OUT + ' — ' + str(round(os.path.getsize(OUT)/1024, 1)) + ' KB')      if os.path.exists(OUT) else '❌ no se generó, copiame el error de arriba')

## Celda 3B — Si dio `CUDA out of memory` (parche)Solo corré esta celda **si la Celda 3 falló por memoria**. Baja la resolución a 192 (menos VRAM). Si aun así falla, `Entorno de ejecución → Reiniciar sesión` y corré Celda 3B directo.

In [ ]:
import os, torchos.chdir('/content/Hunyuan3D-2')from PIL import Imagefrom hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipelinepipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained('tencent/Hunyuan3D-2')img = Image.open(IMG).convert('RGBA')mesh = pipe(image=img, num_inference_steps=30, octree_resolution=192,            generator=torch.manual_seed(0))[0]OUT = '/content/hunyuan_mesh.glb'mesh.export(OUT)print('✅', OUT, round(os.path.getsize(OUT)/1024,1), 'KB' if os.path.exists(OUT) else 'no se generó')

## Celda 4 — Descargar el `.glb`

In [ ]:
from google.colab import filesfiles.download('/content/hunyuan_mesh.glb')

---### Resumen de parches (por si vuelve a fallar)- **`AttributeError _blas_supports_fpe`** → Celda 1B (`numpy>=2.1`) + **Reiniciar sesión**. Es el error más común: pasa si te saltás el reinicio.- **`Model path not exists` / baja 0 files** → estás usando el `mini`. Usá `tencent/Hunyuan3D-2` (Celda 3).- **Sale deforme / con el fondo pegado** → la imagen entró con fondo. Corré la Celda 2B (o subí un PNG transparente).- **`CUDA out of memory`** → Celda 3B (octree 192) o reiniciar sesión.- **Warnings `numba/cuml/cudf`** → inofensivos, ignoralos.- Esto genera **solo la forma** (malla gris, sin color). Para color/textura es un paso extra más pesado (avisá y lo armamos).- Cuando tengas el `.glb`, pasámelo y le pongo las animaciones mocap con el skinning mejorado. Cualquier error rojo, copiámelo. 🐉